# ✈️ Predicting Airline Satisfaction — LightGBM + XGBoost + CatBoost + community stack

**Goal:** for each passenger in the test set, predict the probability that they were *satisfied*.
The leaderboard uses **ROC AUC**, which only cares about how well we *rank* satisfied passengers above unsatisfied ones (not about calibrated probabilities).

**Plan**
1. Load the competition data (+ the original Airline Satisfaction dataset if it's attached as an input — extra real data usually helps in Playground competitions).
2. Engineer a few features that summarise the passenger's experience (average rating, number of bad ratings, total delay, …).
3. Train three different gradient-boosted tree models with 5-fold cross-validation.
4. Blend them, choosing weights that maximise out-of-fold AUC (saved as `submission_own_models.csv`).
5. Stack our models with the community's shared out-of-fold predictions and write the final `submission.csv`.

## 1. Setup
Import libraries and check whether a GPU is available (XGBoost and CatBoost train much faster on GPU; the notebook still works on CPU).

In [1]:
import os, glob, warnings
import numpy as np, pandas as pd
import lightgbm as lgb, xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata
warnings.filterwarnings('ignore')

GPU = os.system('nvidia-smi > /dev/null 2>&1') == 0
SEED, N_FOLDS, TARGET = 42, 5, 'satisfaction'
print('GPU available:', GPU)

GPU available: False


## 2. Load the data
* **Competition files** – `train.csv` (with the target), `test.csv`, `sample_submission.csv`. These are *synthetic* rows generated from a real dataset.
* **Original dataset** – any other attached CSV that has a `satisfaction` column is treated as the real source data. Its labels are text (`satisfied` / `neutral or dissatisfied`), so we convert them to 1/0.

We'll only ever add the original rows to the **training** side of each fold — never to validation — so our CV score stays honest.

In [2]:
csvs = glob.glob('/kaggle/input/**/*.csv', recursive=True)
comp = {os.path.basename(p): p for p in csvs if 'playground-series-s6e10' in p}
train = pd.read_csv(comp['train.csv'])
test = pd.read_csv(comp['test.csv'])
sample = pd.read_csv(comp['sample_submission.csv'])
train[TARGET] = train[TARGET].astype(int)

orig_parts = [pd.read_csv(p) for p in csvs if 'playground-series-s6e10' not in p]
orig_parts = [d for d in orig_parts if TARGET in d.columns]
if orig_parts:
    orig = pd.concat(orig_parts, ignore_index=True)
    orig[TARGET] = orig[TARGET].astype(str).str.strip().str.lower().isin(['satisfied', 'true', '1', '1.0']).astype(int)
    keep = [c for c in train.columns if c != 'id']
    missing = [c for c in keep if c not in orig.columns]
    if missing:
        print('Original data is missing columns, not using it:', missing); orig = None
    else:
        orig = orig[keep].drop_duplicates().reset_index(drop=True)
else:
    orig = None

print('train', train.shape, '| test', test.shape, '| original', None if orig is None else orig.shape)
print('positive rate train: %.3f' % train[TARGET].mean(), '' if orig is None else '| original: %.3f' % orig[TARGET].mean())
train.head()

train (699635, 23) | test (299844, 22) | original (129880, 22)
positive rate train: 0.444 | original: 0.434


,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,...,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,Gender,Customer Type,Type of Travel,Class,satisfaction
0,0,36,694,4,4,4,3,1,4,1,...,4,3,1,0,0.0,Female,Loyal Customer,Personal Travel,Eco,0
1,1,56,651,5,5,5,5,5,4,4,...,5,4,3,0,0.0,Male,Loyal Customer,Business travel,Business,1
2,2,31,1371,3,4,2,5,3,3,3,...,4,5,3,0,0.0,Female,Loyal Customer,Personal Travel,Eco,0
3,3,10,1616,3,5,3,3,3,3,3,...,3,4,3,0,0.0,Male,Loyal Customer,Personal Travel,Eco,0
4,4,23,481,3,4,3,4,5,3,5,...,5,3,5,5,0.0,Female,disloyal Customer,Business travel,Eco,0


## 3. Quick look at the features
* 4 categorical columns: gender, customer type (loyal / disloyal), type of travel (business / personal), class.
* 14 service ratings on a 0–5 scale (0 means "not applicable").
* Age, flight distance and departure/arrival delays.

In [3]:
CATS = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
NUMS = ['Age', 'Flight Distance', 'Departure Delay in Minutes', 'Arrival Delay in Minutes']
RATINGS = [c for c in train.columns if c not in CATS + NUMS + ['id', TARGET]]
print('Ratings:', RATINGS)
print('\nMissing values:\n', train.isna().sum()[train.isna().sum() > 0])
for c in CATS:
    print('\n', train.groupby(c)[TARGET].agg(['mean', 'size']).round(3))

Ratings: ['Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking', 'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort', 'Inflight entertainment', 'On-board service', 'Leg room service', 'Baggage handling', 'Checkin service', 'Cleanliness']

Missing values:
 Arrival Delay in Minutes    292
dtype: int64

          mean    size
Gender               
Female  0.438  347952
Male    0.449  351683

                     mean    size
Customer Type                   
Loyal Customer     0.495  576990
disloyal Customer  0.201  122645

                   mean    size
Type of Travel                
Business travel  0.584  497441
Personal Travel  0.097  202194

            mean    size
Class                  
Business  0.725  342212
Eco       0.168  327404
Eco Plus  0.242   30019


## 4. Feature engineering
Trees can learn most things on their own, but a few summaries make their job easier:
* **Rating summaries** – mean, std, min, and counts of zero / low (≤2) / perfect (5) ratings. One terrible experience often drives dissatisfaction.
* **Delay features** – total delay, whether the plane made up time in the air, delay relative to distance.
* **Travel × Class** – business travellers in Business class behave very differently from personal travellers in Eco.

Categorical columns get a shared category list across train / test / original so every model sees identical codes.

In [4]:
def make_features(df):
    df = df.copy()
    r = df[RATINGS]
    df['rating_mean'] = r.mean(axis=1)
    df['rating_std'] = r.std(axis=1)
    df['rating_min'] = r.replace(0, np.nan).min(axis=1)
    df['n_zero'] = (r == 0).sum(axis=1)
    df['n_low'] = ((r > 0) & (r <= 2)).sum(axis=1)
    df['n_five'] = (r == 5).sum(axis=1)
    arr = df['Arrival Delay in Minutes'].fillna(df['Departure Delay in Minutes'])
    df['total_delay'] = df['Departure Delay in Minutes'] + arr
    df['delay_gain'] = arr - df['Departure Delay in Minutes']
    df['delay_per_km'] = df['total_delay'] / (df['Flight Distance'] + 1)
    df['travel_class'] = df['Type of Travel'].astype(str) + '_' + df['Class'].astype(str)
    return df

ALL_CATS = CATS + ['travel_class']
train_fe, test_fe = make_features(train), make_features(test)
orig_fe = make_features(orig) if orig is not None else None

### Original-data "lookup" features
The synthetic data was generated *from* the original survey, so the original data's satisfaction rate for each value of a column (e.g. "passengers who rated Online boarding 5 were satisfied 87% of the time") is a powerful hint. We add these lookup rates for every column and for a few important pairs of columns.

For the original rows themselves we compute the rates **out-of-fold** (each row's rate comes from the other 80% of the original data); otherwise a row would "see" its own label and the models would learn to over-trust these features.

In [5]:
from sklearn.model_selection import KFold
BASE_COLS = [c for c in train.columns if c not in ['id', TARGET]]
PAIRS = [('Type of Travel', 'Class'), ('Online boarding', 'Inflight wifi service'),
         ('Customer Type', 'Type of Travel'), ('Online boarding', 'Type of Travel'),
         ('Inflight entertainment', 'Seat comfort'), ('Class', 'Online boarding')]

def lookup_keys(df):
    norm = lambda s: s.astype(float).astype(str) if pd.api.types.is_numeric_dtype(s) else s.astype(str).str.strip()
    keys = {c: norm(df[c]).reset_index(drop=True) for c in BASE_COLS}
    for a, b in PAIRS:
        keys[f'{a}|{b}'] = keys[a] + '|' + keys[b]
    return keys

if orig is not None:
    prior = orig[TARGET].mean()
    orig_keys = lookup_keys(orig)
    def rate_map(key, labels, smooth=20):
        g = pd.DataFrame({'k': key.values, 'y': labels}).groupby('k')['y'].agg(['sum', 'count'])
        return (g['sum'] + prior * smooth) / (g['count'] + smooth)
    for f, raw in [(train_fe, train), (test_fe, test)]:
        for name, key in lookup_keys(raw).items():
            f['orig_rate_' + name] = key.map(rate_map(orig_keys[name], orig[TARGET].values)).astype(float).fillna(prior).values
    for name, key in orig_keys.items():
        col = np.full(len(orig), prior)
        for tr_i, va_i in KFold(5, shuffle=True, random_state=SEED).split(orig):
            m = rate_map(key.iloc[tr_i], orig[TARGET].values[tr_i])
            col[va_i] = key.iloc[va_i].map(m).astype(float).fillna(prior).values
        orig_fe['orig_rate_' + name] = col
    print('added', sum(c.startswith('orig_rate_') for c in train_fe.columns), 'lookup features')

added 27 lookup features


Next, give all categorical columns a shared set of categories across train / test / original.

In [6]:
for c in ALL_CATS:
    frames = [train_fe, test_fe] + ([orig_fe] if orig_fe is not None else [])
    levels = sorted(set().union(*[set(f[c].astype(str)) for f in frames]))
    dtype = pd.CategoricalDtype(levels)
    for f in frames:
        f[c] = f[c].astype(str).astype(dtype)

FEATS = [c for c in train_fe.columns if c not in ['id', TARGET]]
y = train_fe[TARGET].values
y_orig = orig_fe[TARGET].values if orig_fe is not None else None
print(len(FEATS), 'features')

58 features


## 5. Cross-validation framework
We split the training data into 5 stratified folds. For each fold the model trains on 4/5 (plus the original data) and predicts the held-out 1/5 — these **out-of-fold (OOF)** predictions give an honest estimate of leaderboard AUC. Test predictions are averaged over the 5 fold models.

In [7]:
folds = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(train_fe, y))

def run_cv(name, fit_predict):
    oof, pred = np.zeros(len(train_fe)), np.zeros(len(test_fe))
    X_test = test_fe[FEATS]
    for i, (tr_idx, va_idx) in enumerate(folds):
        X_tr, y_tr = train_fe.iloc[tr_idx][FEATS], y[tr_idx]
        if orig_fe is not None:
            X_tr = pd.concat([X_tr, orig_fe[FEATS]], ignore_index=True)
            y_tr = np.concatenate([y_tr, y_orig])
        X_va, y_va = train_fe.iloc[va_idx][FEATS], y[va_idx]
        p_va, p_te = fit_predict(X_tr, y_tr, X_va, y_va, X_test)
        oof[va_idx] = p_va
        pred += p_te / N_FOLDS
        print(f'{name} fold {i}: AUC {roc_auc_score(y_va, p_va):.5f}')
    print(f'>>> {name} CV AUC: {roc_auc_score(y, oof):.5f}\n')
    return oof, pred

oofs, preds = {}, {}

## 6. Model 1 — LightGBM
Fast, handles categoricals natively. A low learning rate with early stopping lets it pick the right number of trees per fold.

In [ ]:
def fit_lgb(X_tr, y_tr, X_va, y_va, X_te):
    m = lgb.LGBMClassifier(
        n_estimators=10000, learning_rate=0.03, num_leaves=63, min_child_samples=40,
        subsample=0.8, subsample_freq=1, colsample_bytree=0.5, reg_lambda=2.0,
        cat_smooth=10, random_state=SEED, verbose=-1)
    m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], eval_metric='auc',
          callbacks=[lgb.early_stopping(300, verbose=False)])
    return m.predict_proba(X_va)[:, 1], m.predict_proba(X_te)[:, 1]

oofs['lgb'], preds['lgb'] = run_cv('LightGBM', fit_lgb)

LightGBM fold 0: AUC 0.96020
LightGBM fold 1: AUC 0.95893
LightGBM fold 2: AUC 0.96086


## 7. Model 2 — XGBoost
Grows trees level-wise (depth-limited) rather than leaf-wise, so its errors differ from LightGBM's — that diversity is what makes blending work.

In [ ]:
def fit_xgb(X_tr, y_tr, X_va, y_va, X_te):
    m = xgb.XGBClassifier(
        n_estimators=10000, learning_rate=0.03, max_depth=7, min_child_weight=5,
        subsample=0.8, colsample_bytree=0.5, reg_lambda=2.0, max_cat_to_onehot=8,
        tree_method='hist', device='cuda' if GPU else 'cpu', enable_categorical=True,
        eval_metric='auc', early_stopping_rounds=300, random_state=SEED)
    m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    return m.predict_proba(X_va)[:, 1], m.predict_proba(X_te)[:, 1]

oofs['xgb'], preds['xgb'] = run_cv('XGBoost', fit_xgb)

## 8. Model 3 — CatBoost
Uses symmetric trees and ordered target statistics for categoricals — usually strong on this kind of survey data. On CPU this is the slowest model, so it uses fewer iterations with a higher learning rate.

In [ ]:
def to_cat_input(X):
    X = X.copy()
    X[ALL_CATS] = X[ALL_CATS].astype(str)
    return X

def fit_cat(X_tr, y_tr, X_va, y_va, X_te):
    m = CatBoostClassifier(
        iterations=6000 if GPU else 2500, learning_rate=0.05 if GPU else 0.08, depth=7,
        l2_leaf_reg=3, eval_metric='AUC', task_type='GPU' if GPU else 'CPU',
        od_type='Iter', od_wait=300, random_seed=SEED, verbose=0)
    m.fit(to_cat_input(X_tr), y_tr, cat_features=ALL_CATS,
          eval_set=(to_cat_input(X_va), y_va), use_best_model=True)
    return m.predict_proba(to_cat_input(X_va))[:, 1], m.predict_proba(to_cat_input(X_te))[:, 1]

oofs['cat'], preds['cat'] = run_cv('CatBoost', fit_cat)

## 9. Blending
Because AUC only depends on ranking, we convert each model's predictions to **ranks** (0–1) before averaging — this stops one model's probability scale from dominating. We then search a grid of weights and keep the combination with the highest OOF AUC.

In [ ]:
names = list(oofs)
R_oof = np.column_stack([rankdata(oofs[k]) / len(y) for k in names])
R_te = np.column_stack([rankdata(preds[k]) / len(test_fe) for k in names])

for k in names:
    print(f'{k:>4}: {roc_auc_score(y, oofs[k]):.5f}')
print(f'equal-weight blend: {roc_auc_score(y, R_oof.mean(1)):.5f}')

best_auc, best_w = 0, None
grid = np.arange(0, 1.0001, 0.05)
for w1 in grid:
    for w2 in grid:
        if w1 + w2 > 1: continue
        w = np.array([w1, w2, 1 - w1 - w2])
        auc = roc_auc_score(y, R_oof @ w)
        if auc > best_auc: best_auc, best_w = auc, w
print('best weights', dict(zip(names, best_w.round(2))), f'-> blend CV AUC {best_auc:.5f}')

## 10. Write the submission
Predictions are aligned to the `id` order in `sample_submission.csv` and saved as `/kaggle/working/submission.csv`.

In [ ]:
sub = pd.DataFrame({'id': test_fe['id'], TARGET: R_te @ best_w})
sub = sample[['id']].merge(sub, on='id', how='left')
assert sub[TARGET].notna().all() and len(sub) == len(sample)
sub.to_csv('/kaggle/working/submission.csv', index=False)
print(sub.shape)
sub.head()

In [ ]:
sub.to_csv('/kaggle/working/submission_own_models.csv', index=False)
for k in names:
    np.save(f'/kaggle/working/oof_{k}.npy', oofs[k]); np.save(f'/kaggle/working/test_{k}.npy', preds[k])

## 11. Stacking with the community OOF library
Many competitors publish their **out-of-fold (OOF) and test predictions**. Dariush Afshar's dataset *S6E10 OOF library* (CC0) packages:
* `stack247` – a logistic-regression stack of 247 public models (by Chris Deotte, Kratosyan, Sachith7, Mitudru Dutta, goodpjw2008 and many others — see the dataset's `stack247_inputs.csv` for every author). Nested CV ≈ 0.9622, public LB 0.96187.
* 4 newer models (**r, s, t, u**: two RealMLPs, a shallow XGBoost, a CTR CatBoost) that were *not* inputs to `stack247`, so they can be stacked next to it.

All of it was trained on **exactly the same 5-fold split** we use (`StratifiedKFold(5, shuffle=True, random_state=42)` over `train.csv` in file order) — we verify that below — so our own OOF predictions line up row-for-row.

**How stacking works:** each model's predictions become input features for a simple logistic regression (on the log-odds scale). To keep the score honest, the stacker is fitted on 4 folds and scored on the 5th (*nested CV*). We try a few combinations and keep the best.

⚠️ Differences of a few hundred-thousandths are within fold noise (fold-to-fold sd ≈ 0.0005), so we only switch away from `stack247` alone if a combination wins on most folds.

In [ ]:
from sklearn.linear_model import LogisticRegression
lib_files = glob.glob('/kaggle/input/**/stack247_oof.npy', recursive=True)
assert lib_files, 'Attach the dataset dariushafshar/s6e10-golem-oof-library as an input'
LIB = os.path.dirname(lib_files[0])
lg = lambda p: np.log(np.clip(p, 1e-7, 1 - 1e-7) / (1 - np.clip(p, 1e-7, 1 - 1e-7)))

fold_id = np.zeros(len(y), dtype=int)
for k, (_, va_idx) in enumerate(folds):
    fold_id[va_idx] = k
same = (fold_id == np.load(f'{LIB}/folds_seed42.npy')).mean()
print(f'our folds match the library folds on {same:.2%} of rows')
assert same == 1.0

oof_w, test_w = pd.read_parquet(f'{LIB}/oof_wide.parquet'), pd.read_parquet(f'{LIB}/test_wide.parquet')
assert (oof_w['id'].values == train['id'].values).all() and (test_w['id'].values == test['id'].values).all()

members = {'stack247': (np.load(f'{LIB}/stack247_oof.npy'), np.load(f'{LIB}/stack247_test.npy'))}
NEW = ['r_realmlp_auxev', 's_realmlp_raw_auxev_10fold', 't_xgb_d5_auxev', 'u_catboost_ctr_auxev']
for c in NEW:
    members[c] = (oof_w[c].values, test_w[c].values)
for k in names:
    members['ours_' + k] = (oofs[k], preds[k])
for k, (o, _) in members.items():
    print(f'{k:>28}: own OOF AUC {roc_auc_score(y, o):.6f}')

In [ ]:
def nested_stack(keys, C=1.0):
    X = np.column_stack([lg(members[k][0]) for k in keys])
    XT = np.column_stack([lg(members[k][1]) for k in keys])
    oof_s, te_s, fold_aucs = np.zeros(len(y)), np.zeros(len(XT)), []
    for k in range(N_FOLDS):
        tr_m = fold_id != k
        m = LogisticRegression(C=C, max_iter=2000).fit(X[tr_m], y[tr_m])
        oof_s[~tr_m] = m.decision_function(X[~tr_m])
        te_s += m.decision_function(XT) / N_FOLDS
        fold_aucs.append(roc_auc_score(y[~tr_m], oof_s[~tr_m]))
    return roc_auc_score(y, oof_s), np.array(fold_aucs), te_s

OURS = ['ours_' + k for k in names]
candidates = {
    'stack247 alone': ['stack247'],
    'stack247 + library r,s,t,u': ['stack247'] + NEW,
    'stack247 + our 3 models': ['stack247'] + OURS,
    'stack247 + r,s,t,u + ours': ['stack247'] + NEW + OURS,
}
results = {name: nested_stack(keys) for name, keys in candidates.items()}
base_auc, base_folds, _ = results['stack247 alone']
for name, (auc, fa, _) in results.items():
    print(f'{name:>30}: nested AUC {auc:.6f}  gain {auc - base_auc:+.6f}  folds better {(fa > base_folds).sum()}/5')

## 12. Final submission
Pick the combination with the best nested AUC — but only if it beats `stack247` alone on at least 4 of 5 folds (otherwise the "gain" is probably noise). The log-odds are turned back into probabilities and saved as `submission.csv`.

In [ ]:
best_name = 'stack247 alone'
for name, (auc, fa, _) in results.items():
    if (fa > base_folds).sum() >= 4 and auc > results[best_name][0]:
        best_name = name
print('using:', best_name, f'(nested AUC {results[best_name][0]:.6f})')
final = 1 / (1 + np.exp(-results[best_name][2]))
sub = sample[['id']].merge(pd.DataFrame({'id': test['id'], TARGET: final}), on='id', how='left')
assert sub[TARGET].notna().all() and len(sub) == len(sample)
sub.to_csv('/kaggle/working/submission.csv', index=False)
sub.head()

## Ideas for further improvement
* Train each own model with several random seeds / 10 folds and average.
* Add diverse own models (RealMLP, TabPFN on GPU) — models that are *not* already in `stack247` are the ones that can still add something.
* Tune the stacker's regularisation `C`.

**Credits:** community OOF predictions from Dariush Afshar's *S6E10 OOF library* and the many authors whose public models it stacks; original data from Arseniy Shutko's *Airline Satisfaction Data*.